# Perturb-FISH leave-one-perturbation analysis

This notebook evaluates whether SpiderNet generalizes to unseen melanoma-cell perturbations in the Perturb-FISH dataset. For each of 11 NF-κB perturbation genes, it trains SpiderNet and a niche-aware linear regression baseline without melanoma cells carrying the held-out perturbation, then applies the fitted models to the complete spatial dataset.

The counterfactual analysis replaces selected unperturbed melanoma-cell expression profiles with profiles sampled from cells carrying the held-out perturbation. Predicted T-cell gene-expression and aggregated ligand–receptor (LR) log-fold changes are compared with observed changes using Spearman correlation. The notebook also saves reusable summary tables and publication-oriented comparison and per-perturbation scatter plots.

**Major inputs:** processed SpiderNet bundles produced by `spidernet_dataloading_MIdimselection_PerturbFISH` and its leave-one-perturbation workflow, plus an optional precomputed `celcomen_summary.csv`.

**Major outputs:** fitted SpiderNet and linear models, per-perturbation prediction metrics, observed and predicted log-fold-change tables, seed records, and comparison figures under the configured result directory.



## 0. Dataset setup

In [ ]:
import os
from pathlib import Path
import numpy as np
from run_benchmarks import DATA_ROOT, UPSTREAM_ROOT, PROCESSED_ROOT, OUTPUT_DIR, RESULTS_DIR
OUTPUT_ROOT = UPSTREAM_ROOT
PROCESSED_DATA_ROOT = PROCESSED_ROOT

# Keep species consistent across all stages.
SPECIES = "human"  # "mouse" or "human"

# Perturbation genes for leave-one-perturbation analysis
PERTURB_GENE_OI = [
    "CHUK", "IRAK1", "TRAM1", "LBP", "IRAK4", "PELI1",
    "TAB2", "MAP2K2", "MAP2K6", "IRF7", "MYD88"
]

# These preprocessing parameters are recorded for reference.
# The processed bundles are expected to have already been generated by
# `spidernet_dataloading_MIdimselection_PerturbFISH` (full data) and the
# corresponding leave-one-out preprocessing workflow using the same settings.
N_HVG = 1000
N_HVG_LR = 2000
NUM_NEIGHBORS = 10

# Training parameters
DIM_ENVIR = 23      # number of latent MI dimensions
N_JOBS = 10         # number of parallel CPU cores for initialization and training
MAX_EPOCH = 20000   # maximum number of training epochs

VERSION = "V1"


## 1. Imports and device setup

In [ ]:
import copy
import json
import random
import time

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
import torch
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from torch_scatter import scatter_mean

from SpiderNet.utils import *
from SpiderNet.config import *
from SpiderNet.io import load_processed_data
from SpiderNet.api import build_model, run_training

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Shared paths, configs, and perturbation-gene list

In [ ]:
# ============================================================
# Build paths and configs shared across all stages
# ============================================================
paths = PathConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    version=VERSION,
    species=SPECIES,
)

preprocess_cfg = PreprocessConfig(
    n_hvg=N_HVG,
    n_hvg_lr=N_HVG_LR,
    num_neighbors=NUM_NEIGHBORS,
)

train_cfg = TrainingConfig(
    version=VERSION,
    dim_envir=DIM_ENVIR,
    max_epoch=MAX_EPOCH,
    n_jobs=N_JOBS,
)

processed_data_root = Path(PROCESSED_DATA_ROOT)

base_run_dir = OUTPUT_DIR
base_run_dir.mkdir(parents=True, exist_ok=True)
base_run_dirs = {"run_dir": base_run_dir, "model_dir": base_run_dir / "Model"}

print(base_run_dirs)

with open(base_run_dir / "run_dirs.json", "w", encoding="utf-8") as handle:
    json.dump({k: str(v) for k, v in base_run_dirs.items()}, handle, indent=2)

rows = [
    {"item": "DATA_ROOT", "value": str(DATA_ROOT), "ok": Path(DATA_ROOT).exists()},
    {"item": "OUTPUT_ROOT", "value": str(OUTPUT_ROOT), "ok": Path(OUTPUT_ROOT).exists()},
    {"item": "PROCESSED_DATA_ROOT", "value": str(PROCESSED_DATA_ROOT), "ok": Path(PROCESSED_DATA_ROOT).exists()},
    {"item": "number of perturbation genes", "value": len(PERTURB_GENE_OI), "ok": len(PERTURB_GENE_OI) > 0},
]
check_df = pd.DataFrame(rows)
display(check_df)

if Path(DATA_ROOT).exists():
    top_level_items = sorted([p.name for p in Path(DATA_ROOT).iterdir()])
    print("Top-level items under DATA_ROOT:")
    print(top_level_items[:30])
else:
    print("DATA_ROOT does not exist yet.")

perturb_gene_OI = list(PERTURB_GENE_OI)
perturb_gene_OI_path = base_run_dir / "perturb_gene_OI.npy"
np.save(perturb_gene_OI_path, np.array(perturb_gene_OI))
print(f"Saved perturbation gene list to: {perturb_gene_OI_path}")

processed_data_info = {
    "processed_data_root": str(processed_data_root),
    "base_processed_data_dir": str(processed_data_root),
    "leaveoneout_processed_dir_pattern": str(processed_data_root / "Leaveoneout_<gene>"),
}
with open(base_run_dir / "processed_data_source.json", "w", encoding="utf-8") as handle:
    json.dump(processed_data_info, handle, indent=2)

preprocess_cfg, train_cfg


## 3. Check the processed PerturbFISH SpiderNet bundles

This notebook assumes that the full processed dataset is stored under `PROCESSED_DATA_ROOT` and that each leave-one-out split is stored under `PROCESSED_DATA_ROOT / f"Leaveoneout_{gene}"`.


In [ ]:
if not processed_data_root.exists():
    raise FileNotFoundError(
        f"Processed data root does not exist: {processed_data_root}\n"
        "Please generate the processed PerturbFISH bundles first or update PROCESSED_DATA_ROOT."
    )

required_files = [
    "adata_all.h5ad",
    "adata_list.pkl",
    "SpiderNet_data_pyg_list.pkl",
    "LR_list.pkl",
]

base_missing = [
    file_name for file_name in required_files
    if not (processed_data_root / file_name).exists()
]
if base_missing:
    raise FileNotFoundError(
        "The base processed bundle is incomplete under "
        f"{processed_data_root}. Missing files: {base_missing}"
    )

split_rows = []
for perturb_gene in perturb_gene_OI:
    gene_processed_dir = processed_data_root / f"Leaveoneout_{perturb_gene}"
    missing_files = [
        file_name for file_name in required_files
        if not (gene_processed_dir / file_name).exists()
    ]
    split_rows.append({
        "perturb_gene": perturb_gene,
        "processed_dir": str(gene_processed_dir),
        "exists": gene_processed_dir.exists(),
        "missing_files": "; ".join(missing_files),
        "ready": gene_processed_dir.exists() and len(missing_files) == 0,
    })

split_check_df = pd.DataFrame(split_rows)
display(split_check_df)

if not split_check_df["ready"].all():
    missing_dirs = split_check_df.loc[~split_check_df["ready"], ["perturb_gene", "processed_dir", "missing_files"]]
    raise FileNotFoundError(
        "Some leave-one-out processed bundles are missing or incomplete.\n"
        f"{missing_dirs.to_string(index=False)}"
    )

print(f"Using base processed data from: {processed_data_root}")


## 4. Shared helper functions

In [ ]:
# ============================================================
# Reproducibility configuration
# ------------------------------------------------------------
# GLOBAL_RANDOM_SEED controls notebook-level randomness.
# INSILICO_RANDOM_SEED controls in-silico cancer-cell replacement.
# Each perturbation gene uses INSILICO_RANDOM_SEED + gene_index,
# so rerunning the same analysis gives the same sampled replacement cells.
# ============================================================
GLOBAL_RANDOM_SEED = 42
INSILICO_RANDOM_SEED = GLOBAL_RANDOM_SEED + 10000


def set_seed(seed=GLOBAL_RANDOM_SEED):
    """
    Set random seeds for Python, NumPy, and PyTorch.
    In-silico perturbation sampling additionally uses a local NumPy
    Generator inside apply_in_silico_replacement().
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sanity_check_processed(processed):
    if processed.adata_list is None or processed.spidernet_data is None:
        return None, None

    check_df = pd.DataFrame(
        {
            "n_cells": [processed.adata_list[i].obs.shape[0] for i in range(len(processed.adata_list))],
            "edge_index_max": [
                torch.max(processed.spidernet_data[i]["edge_index"]).cpu().item()
                for i in range(len(processed.adata_list))
            ],
        }
    )
    mismatch = check_df["n_cells"] != (check_df["edge_index_max"] + 1)
    return check_df, mismatch


set_seed(GLOBAL_RANDOM_SEED)
print(f"GLOBAL_RANDOM_SEED = {GLOBAL_RANDOM_SEED}")
print(f"INSILICO_RANDOM_SEED = {INSILICO_RANDOM_SEED}")


## 5. Train leave-one-out SpiderNet models

In [ ]:
time_start_total = time.time()
training_summary = []

for perturb_gene in perturb_gene_OI:
    print(f"\nProcessing leave-one-out model for gene: {perturb_gene}")

    gene_processed_dir = processed_data_root / f"Leaveoneout_{perturb_gene}"
    gene_run_dir = base_run_dir / f"Leaveoneout_{perturb_gene}"
    gene_run_dir.mkdir(parents=True, exist_ok=True)
    gene_model_dir = gene_run_dir / "Model"
    gene_model_dir.mkdir(parents=True, exist_ok=True)

    processed = load_processed_data(gene_processed_dir)

    print("Processed data dir:", gene_processed_dir)
    print("Number of batches:", len(processed.spidernet_data))
    print("Number of LR pairs:", len(processed.lr_list))
    print("Number of training genes:", processed.genenames_train.shape[0])

    check_df, mismatch = sanity_check_processed(processed)
    if mismatch.any():
        print("Potential data-loading mismatch detected:")
        display(check_df.loc[mismatch])
    else:
        print("Sanity check passed.")

    model = build_model(
        processed=processed,
        train_cfg=train_cfg,
        device=device,
    )

    model = run_training(
        model=model,
        processed=processed,
        train_cfg=train_cfg,
        model_dir=gene_model_dir,
        device=device,
    )

    with open(gene_model_dir / "SpiderNet_model_config.json", "w", encoding="utf-8") as handle:
        json.dump(train_cfg.to_dict(), handle, indent=2)

    with open(gene_model_dir / "SpiderNet_preprocess_config.json", "w", encoding="utf-8") as handle:
        json.dump(preprocess_cfg.to_dict(), handle, indent=2)

    with open(gene_run_dir / "processed_data_source.json", "w", encoding="utf-8") as handle:
        json.dump({"processed_data_dir": str(gene_processed_dir)}, handle, indent=2)

    final_model_path = gene_model_dir / f"model_epoch{train_cfg.max_epoch - 1}.pth"
    training_summary.append(
        {
            "perturb_gene": perturb_gene,
            "processed_data_dir": str(gene_processed_dir),
            "run_dir": str(gene_run_dir),
            "model_dir": str(gene_model_dir),
            "final_model_exists": final_model_path.exists(),
            "n_batches": len(processed.spidernet_data),
            "n_lr": len(processed.lr_list),
            "n_genes": int(processed.genenames_train.shape[0]),
        }
    )

time_end_total = time.time()
print(f"\nTotal modeling time cost: {(time_end_total - time_start_total) / 60:.2f} minutes")

training_summary_df = pd.DataFrame(training_summary)
training_summary_df

training_summary_path = base_run_dir / "training_summary_leaveoneout.csv"
training_summary_df.to_csv(training_summary_path, index=False)
print(f"Saved summary to: {training_summary_path}")


## 6. Train leave-one-out linear baseline models

In [ ]:
time_start_total = time.time()
linear_summary = []

for perturb_gene_OI_cur in perturb_gene_OI:
    print("Processing modeling for leave-one-out gene:", perturb_gene_OI_cur)

    gene_processed_dir = processed_data_root / f"Leaveoneout_{perturb_gene_OI_cur}"
    gene_run_dir = base_run_dir / f"Leaveoneout_{perturb_gene_OI_cur}"
    gene_model_dir = gene_run_dir / "Linearbaseline"
    gene_model_dir.mkdir(parents=True, exist_ok=True)

    processed = load_processed_data(gene_processed_dir)

    adata_copy = processed.adata_all
    SpiderNet_data_pyg_list = processed.spidernet_data
    LR_list = processed.lr_list

    if adata_copy is None:
        raise FileNotFoundError(f"Missing adata_all.h5ad under {gene_processed_dir}")
    if SpiderNet_data_pyg_list is None:
        raise FileNotFoundError(f"Missing SpiderNet_data_pyg_list.pkl under {gene_processed_dir}")
    if LR_list is None:
        raise FileNotFoundError(f"Missing LR_list.pkl under {gene_processed_dir}")

    check_df, mismatch = sanity_check_processed(processed)
    if check_df is not None:
        if mismatch.any():
            print("Potential data-loading mismatch detected:")
            print(check_df.loc[mismatch])
        else:
            print("Sanity check passed.")

    cell_class_onehot = SpiderNet_data_pyg_list[0]["cell_class_onehot"].detach().cpu().numpy()
    edge_index = SpiderNet_data_pyg_list[0]["edge_index"]

    edge_index_np = edge_index.detach().cpu().numpy()
    if edge_index_np.shape[0] == 2 and edge_index_np.shape[1] != 2:
        edge_index_np = edge_index_np.T

    cellpair_LRpair_neigh = np.zeros((edge_index_np.shape[0], len(LR_list)))

    for LR_idx, LR_pair in enumerate(LR_list):
        ligand_idx = np.where(np.isin(SpiderNet_data_pyg_list[0]["genenames"], LR_pair[0]))[0]
        exp_ligand = np.array(SpiderNet_data_pyg_list[0]["x"][:, ligand_idx])
        exp_ligand = (
            np.power(np.prod(exp_ligand, axis=1), 1 / exp_ligand.shape[1])
            if exp_ligand.shape[1] > 1 else exp_ligand[:, 0]
        )

        receptor_idx = np.where(np.isin(SpiderNet_data_pyg_list[0]["genenames"], LR_pair[1]))[0]
        exp_receptor = np.array(SpiderNet_data_pyg_list[0]["x"][:, receptor_idx])
        exp_receptor = (
            np.power(np.prod(exp_receptor, axis=1), 1 / exp_receptor.shape[1])
            if exp_receptor.shape[1] > 1 else exp_receptor[:, 0]
        )

        cellpair_LRpair_neigh[:, LR_idx] = np.sqrt(
            exp_ligand[edge_index_np[:, 0]] * exp_receptor[edge_index_np[:, 1]]
        )

    LRcoexp_receiver_agg = scatter_mean(
        torch.tensor(cellpair_LRpair_neigh).to(device),
        torch.tensor(edge_index_np[:, 1]).to(torch.int64).to(device),
        dim=0,
        dim_size=cell_class_onehot.shape[0],
    ).detach().cpu().numpy()

    LRcoexp_sender_agg = scatter_mean(
        torch.tensor(cellpair_LRpair_neigh).to(device),
        torch.tensor(edge_index_np[:, 0]).to(torch.int64).to(device),
        dim=0,
        dim_size=cell_class_onehot.shape[0],
    ).detach().cpu().numpy()

    LRcoexp_sender_receiver_agg = np.hstack([LRcoexp_sender_agg, LRcoexp_receiver_agg])

    X_target = adata_copy.X
    n_cells, n_genes = X_target.shape

    if sp.issparse(X_target):
        y_all = X_target.toarray()
    else:
        y_all = np.asarray(X_target)

    y_all = y_all.astype(np.float32)

    edge = SpiderNet_data_pyg_list[0].edge_index.detach().cpu().numpy()
    if edge.shape[0] == 2 and edge.shape[1] != 2:
        edge = edge.T

    senders = edge[:, 0].astype(np.int64)
    receivers = edge[:, 1].astype(np.int64)

    X_feat_source = adata_copy.X
    if not sp.issparse(X_feat_source):
        X_feat_source = sp.csr_matrix(X_feat_source)
    else:
        X_feat_source = X_feat_source.tocsr()

    rows = np.concatenate([receivers, senders])
    cols = np.concatenate([senders, receivers])
    data = np.ones(len(rows), dtype=np.float32)

    A = sp.coo_matrix((data, (rows, cols)), shape=(n_cells, n_cells)).tocsr()
    A.setdiag(0)
    A.eliminate_zeros()

    sum_all = A @ X_feat_source
    deg = np.asarray(A.sum(axis=1)).ravel()
    deg[deg == 0] = 1.0

    X_feat_all = sum_all.multiply(1.0 / deg[:, None]).toarray().astype(np.float32)

    _, n_types = cell_class_onehot.shape
    cell_type_idx = cell_class_onehot.argmax(axis=1).astype(np.int64)

    senders = edge_index_np[:, 0].astype(np.int64)
    receivers = edge_index_np[:, 1].astype(np.int64)

    sender_type = cell_type_idx[senders]
    receiver_type = cell_type_idx[receivers]
    flat_col = sender_type * n_types + receiver_type

    cellclass_interaction_matrix_flat = np.zeros((n_cells, n_types * n_types), dtype=np.uint8)
    np.maximum.at(cellclass_interaction_matrix_flat, (receivers, flat_col), 1)

    X_feat = np.hstack([
        X_feat_all,
        cell_class_onehot,
        cellclass_interaction_matrix_flat,
        LRcoexp_sender_receiver_agg,
    ]).astype(np.float32)

    print("X_feat shape:", X_feat.shape)

    model_lm = LinearRegression()
    model_lm.fit(X_feat, y_all)

    lm_path = gene_model_dir / f"LinearRegression_model_Tcell_geneexp_prediction_leaveoneout_{perturb_gene_OI_cur}.joblib"
    joblib.dump(model_lm, lm_path)

    y_pred = model_lm.predict(X_feat)
    gene_corr = np.array([spearmanr(y_pred[:, i], y_all[:, i]).correlation for i in range(n_genes)])
    linear_summary.append(
        {
            "perturb_gene": perturb_gene_OI_cur,
            "processed_data_dir": str(gene_processed_dir),
            "linear_model_path": str(lm_path),
            "mean_gene_spearman": np.nanmean(gene_corr),
            "n_features": int(X_feat.shape[1]),
        }
    )

time_end_total = time.time()
print(f"\nTotal linear modeling time cost: {(time_end_total - time_start_total) / 60:.2f} minutes")

linear_summary_df = pd.DataFrame(linear_summary)
linear_summary_path = base_run_dir / "linear_summary_leaveoneout.csv"
linear_summary_df.to_csv(linear_summary_path, index=False)
print(f"Saved linear-model summary to: {linear_summary_path}")

linear_summary_df


## 7. Load shared processed data for in-silico perturbation analysis

In [ ]:
# ============================================================
# Load the base processed dataset used for downstream
# in-silico perturbation analysis
# ============================================================
base_processed = load_processed_data(processed_data_root)

if base_processed.adata_list is None:
    raise FileNotFoundError(f"Missing adata_list.pkl under {processed_data_root}")
if base_processed.spidernet_data is None:
    raise FileNotFoundError(f"Missing SpiderNet_data_pyg_list.pkl under {processed_data_root}")
if base_processed.lr_list is None:
    raise FileNotFoundError(f"Missing LR_list.pkl under {processed_data_root}")

LR_list = base_processed.lr_list

check_df, mismatch = sanity_check_processed(base_processed)
if check_df is not None:
    if mismatch.any():
        print("Potential data-loading mismatch detected in base processed data:")
        print(check_df.loc[mismatch])
    else:
        print("Sanity check passed for base processed data.")


## 8. Helper functions for in-silico spatial perturbation

In [ ]:
def normalize_edge_index(edge_index):
    """
    Convert edge_index to shape (E, 2).
    Supports both (E, 2) and (2, E).
    """
    if torch.is_tensor(edge_index):
        edge_index = edge_index.detach().cpu().numpy()

    edge_index = np.asarray(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index
    if edge_index.shape[0] == 2:
        return edge_index.T

    raise ValueError(f"Unsupported edge_index shape: {edge_index.shape}")


def build_perturbation_annotation(adata_all):
    """
    Build one-hot perturbation annotation for all cells.
    Non-cancer cells are forced to zero.
    """
    perturb_series = adata_all.obs["perturbation"].astype(str).fillna("")
    perturb_annotation = perturb_series.str.get_dummies(sep="_")

    drop_tokens = {"Control", "nan", "None", "none", ""}
    cols_to_keep = [c for c in perturb_annotation.columns if c not in drop_tokens]
    perturb_annotation = perturb_annotation[cols_to_keep]

    perturb_annotation.index = adata_all.obs.index
    perturb_annotation = perturb_annotation.astype("int8")

    is_cancer = adata_all.obs["celltype2"].astype(str).eq("cancer").values
    perturb_annotation.loc[~is_cancer, :] = 0
    return perturb_annotation


def to_gene_str(x, sep="|"):
    """
    Convert ligand/receptor gene containers into a readable string.
    """
    while isinstance(x, (list, tuple)) and len(x) == 1 and isinstance(x[0], (list, tuple)):
        x = x[0]

    if isinstance(x, str):
        return x

    if isinstance(x, (list, tuple)):
        flat = []
        for e in x:
            if isinstance(e, (list, tuple)):
                flat.extend(list(e))
            else:
                flat.append(e)
        flat = [str(e) for e in flat]
        return sep.join(flat)

    return str(x)


def build_lr_feature_names(LR_list):
    """
    Build LR feature names and sender/receiver aggregated feature names.
    """
    lr_merge = [f"{to_gene_str(lr[0])}_{to_gene_str(lr[1])}" for lr in LR_list]
    lr_merge_sr = [f"{x}_S" for x in lr_merge] + [f"{x}_R" for x in lr_merge]
    return lr_merge, lr_merge_sr


def compute_lr_sender_receiver_agg(expr_matrix, edge_index, genenames, LR_list, device):
    """
    Compute LR coexpression on edges, then aggregate to sender/receiver cell level.
    Returns:
        cellpair_LRpair_neigh: (E, n_lr)
        lr_sender_receiver_agg: (n_cells, 2*n_lr)
    """
    if torch.is_tensor(expr_matrix):
        expr_matrix = expr_matrix.detach().cpu().numpy()

    edge_index_np = normalize_edge_index(edge_index)
    expr_matrix = np.asarray(expr_matrix)

    n_edges = edge_index_np.shape[0]
    n_cells = expr_matrix.shape[0]
    cellpair_LRpair_neigh = np.zeros((n_edges, len(LR_list)), dtype=np.float32)

    for LR_idx, LR_pair in enumerate(LR_list):
        ligand_idx = np.where(np.isin(genenames, LR_pair[0]))[0]
        receptor_idx = np.where(np.isin(genenames, LR_pair[1]))[0]

        exp_ligand = expr_matrix[:, ligand_idx]
        exp_receptor = expr_matrix[:, receptor_idx]

        exp_ligand = (
            np.power(np.prod(exp_ligand, axis=1), 1 / exp_ligand.shape[1])
            if exp_ligand.shape[1] > 1 else exp_ligand[:, 0]
        )
        exp_receptor = (
            np.power(np.prod(exp_receptor, axis=1), 1 / exp_receptor.shape[1])
            if exp_receptor.shape[1] > 1 else exp_receptor[:, 0]
        )

        cellpair_LRpair_neigh[:, LR_idx] = np.sqrt(
            exp_ligand[edge_index_np[:, 0]] * exp_receptor[edge_index_np[:, 1]]
        )

    edge_index_tensor = torch.as_tensor(edge_index_np, dtype=torch.long, device=device)
    cellpair_tensor = torch.tensor(cellpair_LRpair_neigh, dtype=torch.float32, device=device)

    lr_receiver_agg = scatter_mean(
        cellpair_tensor,
        edge_index_tensor[:, 1].to(torch.int64),
        dim=0,
        dim_size=n_cells,
    ).detach().cpu().numpy()

    lr_sender_agg = scatter_mean(
        cellpair_tensor,
        edge_index_tensor[:, 0].to(torch.int64),
        dim=0,
        dim_size=n_cells,
    ).detach().cpu().numpy()

    lr_sender_receiver_agg = np.hstack([lr_sender_agg, lr_receiver_agg]).astype(np.float32)
    return cellpair_LRpair_neigh, lr_sender_receiver_agg


def build_celltype_interaction_features(edge_index_np, cell_class_onehot):
    """
    Build flattened sender-type x receiver-type interaction indicator for each receiver cell.
    """
    n_cells, n_types = cell_class_onehot.shape
    cell_type_idx = cell_class_onehot.argmax(axis=1).astype(np.int64)

    senders = edge_index_np[:, 0].astype(np.int64)
    receivers = edge_index_np[:, 1].astype(np.int64)

    sender_type = cell_type_idx[senders]
    receiver_type = cell_type_idx[receivers]

    flat_col = sender_type * n_types + receiver_type
    interaction_mat = np.zeros((n_cells, n_types * n_types), dtype=np.uint8)
    np.maximum.at(interaction_mat, (receivers, flat_col), 1)
    return interaction_mat


def build_linear_features(
    expr_matrix,
    edge_index,
    cell_class_onehot,
    lr_sender_receiver_agg,
    include_interaction_features=True,
):
    """
    Build input features for the linear baseline.

    Default feature blocks:
    1) neighbor mean gene expression
    2) cell-type one-hot
    3) sender/receiver cell-type interaction indicators
    4) aggregated LR activity

    Set include_interaction_features=False only for compatibility with
    previously saved linear models that were trained without the
    interaction block.
    """
    edge_index_np = normalize_edge_index(edge_index)

    if torch.is_tensor(expr_matrix):
        expr_matrix = expr_matrix.detach().cpu().numpy()

    X = expr_matrix
    if not sp.issparse(X):
        X = sp.csr_matrix(X)
    else:
        X = X.tocsr()

    n_cells = X.shape[0]
    senders = edge_index_np[:, 0].astype(np.int64)
    receivers = edge_index_np[:, 1].astype(np.int64)

    rows = np.concatenate([receivers, senders])
    cols = np.concatenate([senders, receivers])
    data = np.ones(len(rows), dtype=np.float32)
    A = sp.coo_matrix((data, (rows, cols)), shape=(n_cells, n_cells)).tocsr()

    A.setdiag(0)
    A.eliminate_zeros()

    sum_all = A @ X
    deg = np.asarray(A.sum(axis=1)).ravel()
    deg[deg == 0] = 1.0
    neighbor_mean = sum_all.multiply(1.0 / deg[:, None]).toarray().astype(np.float32)

    feature_blocks = [neighbor_mean, cell_class_onehot]

    if include_interaction_features:
        interaction_mat = build_celltype_interaction_features(edge_index_np, cell_class_onehot)
        feature_blocks.append(interaction_mat)

    feature_blocks.append(lr_sender_receiver_agg)

    X_feat = np.hstack(feature_blocks).astype(np.float32)
    return X_feat


def compute_tcell_neighboring_perturbation_matrix(
    adata_all,
    edge_index,
    perturb_annotation,
    perturb_gene_OI,
):
    """
    For each T cell, count whether it has neighboring perturbed cancer cells for each perturbation gene.
    """
    edge_index_np = normalize_edge_index(edge_index)
    Tcell_index = np.where(adata_all.obs["celltype2"] == "T cells")[0]

    neighboring_counts = np.zeros((len(Tcell_index), len(perturb_gene_OI)), dtype=np.float32)

    for row_idx, Tcell_index_cur in enumerate(Tcell_index):
        neighbors_all = edge_index_np[np.where(edge_index_np[:, 1] == Tcell_index_cur)[0], 0]
        neighbors_perturbation = perturb_annotation.iloc[neighbors_all, :]
        for i, pg in enumerate(perturb_gene_OI):
            neighboring_counts[row_idx, i] = np.sum(neighbors_perturbation[pg])

    neighboring_binary = (neighboring_counts > 0).astype(int)
    return Tcell_index, neighboring_binary


def compute_golden_lfc(
    adata_all,
    Tcell_index,
    neighboring_binary,
    perturb_gene_OI,
    gene_names,
    lr_sender_receiver_agg_ori,
    lr_merge_sr,
):
    """
    Compute observed (golden) LFC for T-cell gene expression and aggregated LR features.
    """
    LFC_Tcell_dict = {}
    LFC_LRagg_dict = {}

    for i, pg in enumerate(perturb_gene_OI):
        near_mask = neighboring_binary[:, i] == 1
        notnear_mask = neighboring_binary[:, i] == 0

        Tcell_index_near = Tcell_index[np.where(near_mask)[0]]
        Tcell_index_notnear = Tcell_index[np.where(notnear_mask)[0]]

        meanexp_near = np.mean(adata_all[Tcell_index_near].X, axis=0)
        meanexp_notnear = np.mean(adata_all[Tcell_index_notnear].X, axis=0)
        lfc_gene = np.log2(meanexp_near / meanexp_notnear)
        LFC_Tcell_dict[pg] = pd.DataFrame(lfc_gene, index=gene_names, columns=[pg])

        meanagg_near = np.mean(lr_sender_receiver_agg_ori[Tcell_index_near, :], axis=0)
        meanagg_notnear = np.mean(lr_sender_receiver_agg_ori[Tcell_index_notnear, :], axis=0)
        lfc_lr = np.log2(meanagg_near / meanagg_notnear)
        LFC_LRagg_dict[pg] = pd.DataFrame(lfc_lr, index=lr_merge_sr, columns=[pg])

    LFC_Tcell_golden = pd.concat([LFC_Tcell_dict[pg] for pg in perturb_gene_OI], axis=1)
    LFC_LRagg_golden = pd.concat([LFC_LRagg_dict[pg] for pg in perturb_gene_OI], axis=1)
    return LFC_Tcell_golden, LFC_LRagg_golden


def predict_batches_spidernet_and_linear(model, model_lm, data_list, LR_list, device):
    """
    Run SpiderNet and linear baseline on all batches.
    Returns concatenated reconstructed gene expression and LR-aggregated features.
    """
    exp_recon = []
    exp_recon_linear = []
    exp_LR_agg_recon = []
    exp_LR_agg_recon_linear = []

    for data_cur in data_list:
        cellclass_onehot_cur = data_cur["cell_class_onehot"].detach().cpu().numpy()
        edge_index_cur = data_cur["edge_index"]
        edge_index_np = normalize_edge_index(edge_index_cur)
        genenames_cur = np.array(data_cur["genenames"])

        # SpiderNet prediction
        exp_reconcur, exp_LR_reconcur, _, _, _, _, _, _ = model(data_cur.to(device))
        exp_reconcur = exp_reconcur.detach().cpu().numpy()
        exp_recon.append(exp_reconcur)

        edge_index_tensor = torch.as_tensor(edge_index_np, dtype=torch.long, device=device)
        exp_LR_tensor = torch.tensor(exp_LR_reconcur, dtype=torch.float32, device=device)

        exp_LR_receiver = scatter_mean(
            exp_LR_tensor,
            edge_index_tensor[:, 1].to(torch.int64),
            dim=0,
            dim_size=cellclass_onehot_cur.shape[0],
        ).detach().cpu().numpy()

        exp_LR_sender = scatter_mean(
            exp_LR_tensor,
            edge_index_tensor[:, 0].to(torch.int64),
            dim=0,
            dim_size=cellclass_onehot_cur.shape[0],
        ).detach().cpu().numpy()

        exp_LR_agg_recon.append(np.hstack([exp_LR_sender, exp_LR_receiver]).astype(np.float32))

        # Linear baseline prediction
        _, lr_sender_receiver_agg_ori = compute_lr_sender_receiver_agg(
            expr_matrix=data_cur["x"],
            edge_index=edge_index_cur,
            genenames=genenames_cur,
            LR_list=LR_list,
            device=device,
        )

        X_feat = build_linear_features(
            expr_matrix=data_cur["x"],
            edge_index=edge_index_cur,
            cell_class_onehot=cellclass_onehot_cur,
            lr_sender_receiver_agg=lr_sender_receiver_agg_ori,
            include_interaction_features=True,
        )

        expected_n_features = getattr(model_lm, "n_features_in_", X_feat.shape[1])
        if X_feat.shape[1] != expected_n_features:
            X_feat_compat = build_linear_features(
                expr_matrix=data_cur["x"],
                edge_index=edge_index_cur,
                cell_class_onehot=cellclass_onehot_cur,
                lr_sender_receiver_agg=lr_sender_receiver_agg_ori,
                include_interaction_features=False,
            )

            if X_feat_compat.shape[1] == expected_n_features:
                print(
                    "Warning: loaded linear baseline was trained without "
                    "cell-type interaction features. Using compatibility mode. "
                    "Re-run Section 6 to retrain the linear baseline with the corrected feature set."
                )
                X_feat = X_feat_compat
            else:
                raise ValueError(
                    "Linear baseline feature mismatch: "
                    f"prediction built {X_feat.shape[1]} features, "
                    f"compatibility mode built {X_feat_compat.shape[1]} features, "
                    f"but the saved model expects {expected_n_features}. "
                    "Please re-run Section 6 to retrain the linear baseline."
                )

        exp_recon_linear_cur = model_lm.predict(X_feat)
        exp_recon_linear_cur[exp_recon_linear_cur < 0] = 0
        exp_recon_linear.append(exp_recon_linear_cur.astype(np.float32))

        _, lr_sender_receiver_agg_linear = compute_lr_sender_receiver_agg(
            expr_matrix=exp_recon_linear_cur,
            edge_index=edge_index_cur,
            genenames=genenames_cur,
            LR_list=LR_list,
            device=device,
        )
        exp_LR_agg_recon_linear.append(lr_sender_receiver_agg_linear.astype(np.float32))

    return (
        np.concatenate(exp_recon, axis=0).astype(np.float32),
        np.concatenate(exp_recon_linear, axis=0).astype(np.float32),
        np.concatenate(exp_LR_agg_recon, axis=0).astype(np.float32),
        np.concatenate(exp_LR_agg_recon_linear, axis=0).astype(np.float32),
    )


def get_control_and_perturbed_cancer_indices(adata_all, perturb_annotation, perturb_gene_OI, perturb_gene_cur):
    """
    Get:
    1) cells perturbed by the current perturbation gene
    2) all perturbed cancer cells from all perturbation genes
    3) control cancer cells without any perturbation in perturb_gene_OI
    """
    idx = adata_all.obs.index[
        adata_all.obs["perturbation"].astype(str).str.contains(perturb_gene_cur)
    ]
    cellindex_perturbgene = np.where(adata_all.obs.index.isin(idx))[0]

    cellindex_all_perturbed = []
    for pg in perturb_gene_OI:
        idx_pg = adata_all.obs.index[
            (adata_all.obs["celltype2"] == "cancer")
            & (adata_all.obs["perturbation"].astype(str).str.contains(pg))
        ]
        cellindex_pg = np.where(adata_all.obs.index.isin(idx_pg))[0]
        cellindex_all_perturbed.extend(cellindex_pg.tolist())

    cellindex_all_perturbed = np.unique(cellindex_all_perturbed)
    cellindex_control = np.setdiff1d(
        np.where(adata_all.obs["celltype2"] == "cancer")[0],
        cellindex_all_perturbed,
    )

    cancercell_perturbed_index = np.where(
        (np.array(adata_all.obs["celltype2"]) == "cancer")
        & (np.array(perturb_annotation[perturb_gene_cur]) == 1)
    )[0]

    return cellindex_perturbgene, cellindex_control, cancercell_perturbed_index


def apply_in_silico_replacement(
    data_list,
    nearcellindex_cancer_cur,
    source_perturbed_index,
    random_seed=None,
):
    """
    Replace selected control cancer cells with sampled perturbed cancer-cell expression profiles.

    Reproducibility note
    --------------------
    The original implementation used np.random.choice(), which depends on the
    global NumPy random state and can change if earlier cells consume random
    numbers. Here we use a local NumPy Generator with an explicit random_seed,
    so the same perturbation gene always uses the same replacement mapping.
    """
    nearcellindex_cancer_cur = np.asarray(nearcellindex_cancer_cur, dtype=np.int64)
    source_perturbed_index = np.asarray(source_perturbed_index, dtype=np.int64)

    data_list_perturbed = [copy.deepcopy(d) for d in data_list]

    if len(nearcellindex_cancer_cur) == 0:
        print("No nearby control cancer cells to replace; returning unchanged copied data_list.")
        return data_list_perturbed

    if len(source_perturbed_index) == 0:
        raise ValueError("source_perturbed_index is empty; cannot perform in-silico replacement.")

    if random_seed is None:
        random_seed = globals().get("INSILICO_RANDOM_SEED", globals().get("GLOBAL_RANDOM_SEED", 123))

    rng = np.random.default_rng(int(random_seed))

    for batch_index_cur in range(len(data_list_perturbed)):
        sampling_index = rng.choice(
            source_perturbed_index,
            size=len(nearcellindex_cancer_cur),
            replace=True,
        )
        data_list_perturbed[batch_index_cur].x[
            nearcellindex_cancer_cur[:, None], :
        ] = data_list[batch_index_cur].x[
            sampling_index[:, None], :
        ]

    return data_list_perturbed

def compute_lfc_from_two_groups(before_mat, after_mat, target_index, feature_names, col_name, eps=1e-8):
    """
    Compute log2 fold change between after and before matrices within selected target cells.
    """
    mean_before = np.mean(before_mat[target_index], axis=0)
    mean_after = np.mean(after_mat[target_index], axis=0)
    lfc = np.log2((mean_after + eps) / (mean_before + eps))
    return pd.DataFrame(lfc, index=feature_names, columns=[col_name])


def evaluate_prediction_against_golden(
    LFC_golden_cur,
    LFC_predicted_cur,
    plot_path=None,
    xlabel="Golden LFC",
    ylabel="Predicted LFC",
    title_prefix="",
):
    """
    Evaluate prediction using Spearman correlation and MSE.
    """
    common_idx = LFC_golden_cur.index.intersection(LFC_predicted_cur.index)
    corr, pvalue = spearmanr(LFC_golden_cur.loc[common_idx], LFC_predicted_cur.loc[common_idx])
    mse = np.mean((LFC_golden_cur.loc[common_idx] - LFC_predicted_cur.loc[common_idx]) ** 2)

    if plot_path is not None:
        plt.figure(figsize=(4, 4))
        plt.scatter(LFC_golden_cur.loc[common_idx], LFC_predicted_cur.loc[common_idx], s=5, alpha=0.5)
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.title(f"{title_prefix}\nCorrelation: {np.round(corr, 3)}")
        plt.axhline(0, color="grey", linestyle="--", linewidth=0.5)
        plt.axvline(0, color="grey", linestyle="--", linewidth=0.5)
        plt.plot([-3, 3], [-3, 3], color="red", linestyle="--", linewidth=0.5)
        sns.regplot(
            x=LFC_golden_cur.loc[common_idx],
            y=LFC_predicted_cur.loc[common_idx],
            scatter=False,
            color="blue",
            line_kws={"linewidth": 0.5},
        )
        plt.tight_layout()
        plt.savefig(plot_path, dpi=300)
        plt.close()

    return corr, pvalue, mse

## 9. Run in-silico spatial perturbation analysis

In [ ]:
# ============================================================
# In-silico spatial perturbation analysis
#
# Goal:
# Evaluate whether SpiderNet and the linear baseline can predict
# the transcriptional and LR-level response of T cells to nearby
# perturbed cancer cells.
#
# Strategy:
# 1) derive observed T-cell response from real perturbation data
# 2) simulate perturbation in silico by replacing nearby control
#    cancer cells with perturbed cancer-cell profiles
# 3) compare predicted response against observed response
# ============================================================

# ============================================================
# Step 0. Initialize summary containers
# Collect per-gene evaluation results for:
# 1) SpiderNet gene-expression prediction
# 2) Linear-baseline gene-expression prediction
# 3) SpiderNet aggregated LR prediction
# 4) Linear-baseline aggregated LR prediction
# ============================================================
set_seed(GLOBAL_RANDOM_SEED)
print(f"Running in-silico analysis with INSILICO_RANDOM_SEED = {INSILICO_RANDOM_SEED}")

insilico_seed_records = []
SpiderNet_summary_list = []
linearmodel_summary_list = []
SpiderNet_summary_LR_list = []
linearmodel_summary_LR_list = []

# Store the full observed/predicted LFC profiles the first time they are computed.
# These tables are reused by the downstream barplot/scatter cells, so the notebook
# no longer needs to reload models and recompute in-silico predictions for plotting.
LFC_predicted_gene_spidernet_list = []
LFC_predicted_gene_linearmodel_list = []
LFC_predicted_lr_spidernet_list = []
LFC_predicted_lr_linearmodel_list = []


# ============================================================
# Step 1. Load the base processed dataset used for downstream
# in-silico perturbation analysis
#
# This "base" dataset provides:
# - the full AnnData object reconstructed from all batches
# - the PyG graph objects used as model input
# - the common gene space shared across all analyses
# ============================================================
adata_list = base_processed.adata_list
SpiderNet_data_pyg_list = base_processed.spidernet_data
adata_all = adata_list[0].concatenate(adata_list[1:])
gene_names = adata_all.var_names


# ============================================================
# Step 2. Build shared annotations and reference features
#
# These quantities are reused across all perturbation genes:
# - perturbation one-hot annotation per cell
# - global edge index
# - LR feature names
# - observed aggregated LR activity from the original data
# ============================================================
perturb_annotation = build_perturbation_annotation(adata_all)
edge_index_all = normalize_edge_index(SpiderNet_data_pyg_list[0].edge_index)
LR_merge, LR_merge_SR = build_lr_feature_names(LR_list)

_, LRcoexp_sender_receiver_agg_ori = compute_lr_sender_receiver_agg(
    expr_matrix=SpiderNet_data_pyg_list[0]["x"],
    edge_index=SpiderNet_data_pyg_list[0]["edge_index"],
    genenames=np.array(SpiderNet_data_pyg_list[0]["genenames"]),
    LR_list=LR_list,
    device=device,
)


# ============================================================
# Step 3. Define T-cell neighborhoods relative to perturbed
# cancer cells in the observed data
#
# For each T cell and each perturbation gene, determine whether
# that T cell is adjacent to at least one cancer cell carrying
# that perturbation.
# ============================================================
Tcell_index, neighboring_binary = compute_tcell_neighboring_perturbation_matrix(
    adata_all=adata_all,
    edge_index=edge_index_all,
    perturb_annotation=perturb_annotation,
    perturb_gene_OI=perturb_gene_OI,
)


# ============================================================
# Step 4. Compute the "golden-standard" LFC from observed data
#
# For each perturbation gene, compare:
# - T cells near perturbed cancer cells
# - T cells not near perturbed cancer cells
#
# This produces the reference LFC for:
# 1) gene expression
# 2) aggregated LR activity
# ============================================================
LFC_Tcell_golden, LFC_LRagg_golden = compute_golden_lfc(
    adata_all=adata_all,
    Tcell_index=Tcell_index,
    neighboring_binary=neighboring_binary,
    perturb_gene_OI=perturb_gene_OI,
    gene_names=gene_names,
    lr_sender_receiver_agg_ori=LRcoexp_sender_receiver_agg_ori,
    lr_merge_sr=LR_merge_SR,
)


# ============================================================
# Step 5. Loop over each perturbation gene
#
# For each leave-one-out perturbation gene:
# 1) load the corresponding trained SpiderNet model
# 2) load the corresponding trained linear baseline
# 3) generate predictions before in-silico perturbation
# 4) perform in-silico replacement of nearby control cancer cells
# 5) generate predictions after perturbation
# 6) compute predicted LFC
# 7) compare predicted LFC with observed golden-standard LFC
# ============================================================
for perturb_gene_cur in perturb_gene_OI:
    print("Processing gene:", perturb_gene_cur)

    # --------------------------------------------------------
    # Step 5.1. Define per-gene input/output directories
    # --------------------------------------------------------
    gene_processed_dir = processed_data_root / f"Leaveoneout_{perturb_gene_cur}"
    gene_run_dir = RESULTS_DIR / f"Leaveoneout_{perturb_gene_cur}"
    gene_model_dir = gene_run_dir / "Model"
    gene_linear_dir = gene_run_dir / "Linearbaseline"

    # --------------------------------------------------------
    # Step 5.2. Load the processed leave-one-out dataset and
    # perform a sanity check against the expected input format
    # --------------------------------------------------------
    processed_gene = load_processed_data(gene_processed_dir)
    check_df, mismatch = sanity_check_processed(processed_gene)
    if check_df is not None:
        if mismatch.any():
            print("Potential data-loading mismatch detected:")
            print(check_df.loc[mismatch])
        else:
            print("Sanity check passed.")

    # --------------------------------------------------------
    # Step 5.3. Load the trained SpiderNet model corresponding
    # to the current leave-one-out perturbation gene
    # --------------------------------------------------------
    model = build_model(processed=processed_gene, train_cfg=train_cfg, device=device)
    model.load_state_dict(
        torch.load(
            gene_model_dir / f"model_epoch{train_cfg.max_epoch - 1}.pth",
            map_location=device,
        )
    )
    model = model.to(device)
    model.eval()

    # --------------------------------------------------------
    # Step 5.4. Load the trained linear baseline model for the
    # current leave-one-out perturbation gene
    # --------------------------------------------------------
    model_lm = joblib.load(
        gene_linear_dir / f"LinearRegression_model_Tcell_geneexp_prediction_leaveoneout_{perturb_gene_cur}.joblib"
    )

    # --------------------------------------------------------
    # Step 5.5. Predict the baseline state before in-silico
    # perturbation using both SpiderNet and the linear baseline
    #
    # Outputs include:
    # - reconstructed gene expression
    # - reconstructed aggregated LR activity
    # --------------------------------------------------------
    exp_recon, exp_recon_linear, exp_LR_agg_recon, exp_LR_agg_recon_linear = \
        predict_batches_spidernet_and_linear(
            model=model,
            model_lm=model_lm,
            data_list=SpiderNet_data_pyg_list,
            LR_list=LR_list,
            device=device,
        )

    # --------------------------------------------------------
    # Step 5.6. Identify:
    # - cells carrying the current perturbation
    # - control cancer cells without any perturbation in the set
    # - perturbed cancer cells that can serve as replacement donors
    # --------------------------------------------------------
    cellindex_perturbgene, cellindex_control, cancercell_perturbed_index = \
        get_control_and_perturbed_cancer_indices(
            adata_all=adata_all,
            perturb_annotation=perturb_annotation,
            perturb_gene_OI=perturb_gene_OI,
            perturb_gene_cur=perturb_gene_cur,
        )

    # --------------------------------------------------------
    # Step 5.7. Split T cells into:
    # - T cells near cancer cells perturbed for the current gene
    # - T cells not near such perturbed cancer cells
    #
    # The latter group will be used as the target population for
    # in-silico perturbation analysis.
    # --------------------------------------------------------
    i = perturb_gene_OI.index(perturb_gene_cur)
    Tcell_index_near = Tcell_index[np.where(neighboring_binary[:, i] == 1)[0]]
    Tcell_index_notnear = Tcell_index[np.where(neighboring_binary[:, i] == 0)[0]]

    # --------------------------------------------------------
    # Step 5.8. Identify control cancer cells located near the
    # target T-cell population. These cells will be replaced by
    # sampled perturbed cancer cells in silico.
    # --------------------------------------------------------
    nearcellindex_cur = [
        edge_index_all[:, 1][np.isin(edge_index_all[:, 0], Tcell_index_notnear)],
        edge_index_all[:, 0][np.isin(edge_index_all[:, 1], Tcell_index_notnear)],
    ]
    nearcellindex_cur = np.unique(np.hstack(nearcellindex_cur))
    nearcellindex_cancer_cur = np.intersect1d(nearcellindex_cur, cellindex_control)

    # --------------------------------------------------------
    # Step 5.9. Perform in-silico perturbation by replacing the
    # expression profiles of nearby control cancer cells with
    # sampled profiles from truly perturbed cancer cells
    # --------------------------------------------------------
    insilico_seed_cur = INSILICO_RANDOM_SEED + int(i)
    insilico_seed_records.append(
        {
            "Gene": perturb_gene_cur,
            "GeneIndex": int(i),
            "InSilicoSeed": int(insilico_seed_cur),
            "N_control_cancer_cells_replaced": int(len(nearcellindex_cancer_cur)),
            "N_perturbed_cancer_donor_cells": int(len(cancercell_perturbed_index)),
        }
    )

    SpiderNet_data_pyg_list_perturbed = apply_in_silico_replacement(
        data_list=SpiderNet_data_pyg_list,
        nearcellindex_cancer_cur=nearcellindex_cancer_cur,
        source_perturbed_index=cancercell_perturbed_index,
        random_seed=insilico_seed_cur,
    )

    # --------------------------------------------------------
    # Step 5.10. Predict the perturbed state after in-silico
    # replacement using both SpiderNet and the linear baseline
    # --------------------------------------------------------
    exp_recon_perturb, exp_recon_linear_perturb, exp_LR_agg_recon_perturb, exp_LR_agg_recon_linear_perturb = \
        predict_batches_spidernet_and_linear(
            model=model,
            model_lm=model_lm,
            data_list=SpiderNet_data_pyg_list_perturbed,
            LR_list=LR_list,
            device=device,
        )

    # --------------------------------------------------------
    # Step 5.11. Compute predicted log-fold changes (LFC)
    # in the target T-cell population after in-silico perturbation
    #
    # LFC is computed for:
    # - gene expression (SpiderNet)
    # - gene expression (linear baseline)
    # - aggregated LR activity (SpiderNet)
    # - aggregated LR activity (linear baseline)
    # --------------------------------------------------------
    LFC_predicted_gene = compute_lfc_from_two_groups(
        before_mat=exp_recon,
        after_mat=exp_recon_perturb,
        target_index=Tcell_index_notnear,
        feature_names=gene_names,
        col_name=perturb_gene_cur,
    )
    LFC_predicted_gene_linear = compute_lfc_from_two_groups(
        before_mat=exp_recon_linear,
        after_mat=exp_recon_linear_perturb,
        target_index=Tcell_index_notnear,
        feature_names=gene_names,
        col_name=perturb_gene_cur,
    )
    LFC_predicted_lr = compute_lfc_from_two_groups(
        before_mat=exp_LR_agg_recon,
        after_mat=exp_LR_agg_recon_perturb,
        target_index=Tcell_index_notnear,
        feature_names=LR_merge_SR,
        col_name=perturb_gene_cur,
    )
    LFC_predicted_lr_linear = compute_lfc_from_two_groups(
        before_mat=exp_LR_agg_recon_linear,
        after_mat=exp_LR_agg_recon_linear_perturb,
        target_index=Tcell_index_notnear,
        feature_names=LR_merge_SR,
        col_name=perturb_gene_cur,
    )

    # --------------------------------------------------------
    # Step 5.12. Store the full per-feature LFC profiles.
    #
    # These are the exact LFC tables needed by downstream
    # visualization cells. Saving them here avoids rerunning
    # model prediction and LFC computation later.
    # --------------------------------------------------------
    LFC_predicted_gene_spidernet_list.append(LFC_predicted_gene)
    LFC_predicted_gene_linearmodel_list.append(LFC_predicted_gene_linear)
    LFC_predicted_lr_spidernet_list.append(LFC_predicted_lr)
    LFC_predicted_lr_linearmodel_list.append(LFC_predicted_lr_linear)

    # --------------------------------------------------------
    # Step 5.13. Compare predicted LFC with the observed
    # golden-standard LFC
    #
    # Evaluation metrics:
    # - Spearman correlation
    # - MSE (for gene-expression prediction)
    # --------------------------------------------------------
    corr_sp, p_sp, mse_sp = evaluate_prediction_against_golden(
        LFC_golden_cur=LFC_Tcell_golden[perturb_gene_cur],
        LFC_predicted_cur=LFC_predicted_gene[perturb_gene_cur],
        plot_path=base_run_dir / f"Scatter_Predicted_vs_Golden_LFC_in_T_cells_{perturb_gene_cur}.png",
        xlabel="Golden LFC in T cells",
        ylabel="Predicted LFC in T cells",
        title_prefix=f"Perturbed gene: {perturb_gene_cur}",
    )

    corr_lm, p_lm, mse_lm = evaluate_prediction_against_golden(
        LFC_golden_cur=LFC_Tcell_golden[perturb_gene_cur],
        LFC_predicted_cur=LFC_predicted_gene_linear[perturb_gene_cur],
    )

    corr_lr_sp, p_lr_sp, _ = evaluate_prediction_against_golden(
        LFC_golden_cur=LFC_LRagg_golden[perturb_gene_cur],
        LFC_predicted_cur=LFC_predicted_lr[perturb_gene_cur],
    )

    corr_lr_lm, p_lr_lm, _ = evaluate_prediction_against_golden(
        LFC_golden_cur=LFC_LRagg_golden[perturb_gene_cur],
        LFC_predicted_cur=LFC_predicted_lr_linear[perturb_gene_cur],
    )

    # --------------------------------------------------------
    # Step 5.14. Store per-gene summary statistics for later
    # aggregation across perturbation genes
    # --------------------------------------------------------
    SpiderNet_summary = pd.DataFrame(
        {"Correlation": [corr_sp], "MSE": [mse_sp]},
        index=[perturb_gene_cur],
    )
    linearmodel_summary = pd.DataFrame(
        {"Correlation": [corr_lm], "MSE": [mse_lm]},
        index=[perturb_gene_cur],
    )
    SpiderNet_summary_LRagg = pd.DataFrame(
        {"Correlation": [corr_lr_sp]},
        index=[perturb_gene_cur],
    )
    linearmodel_summary_LRagg = pd.DataFrame(
        {"Correlation": [corr_lr_lm]},
        index=[perturb_gene_cur],
    )

    SpiderNet_summary_list.append(SpiderNet_summary)
    linearmodel_summary_list.append(linearmodel_summary)
    SpiderNet_summary_LR_list.append(SpiderNet_summary_LRagg)
    linearmodel_summary_LR_list.append(linearmodel_summary_LRagg)


## 10. Save summary outputs

In [ ]:
# ============================================================
# Save summary outputs and reusable LFC tables
# ------------------------------------------------------------
# The full LFC tables are saved here immediately after the
# first in-silico perturbation analysis. Downstream plotting
# cells should load these files instead of recomputing LFCs.
# ============================================================

SpiderNet_summary_all = pd.concat(SpiderNet_summary_list, axis=0)
linearmodel_summary_all = pd.concat(linearmodel_summary_list, axis=0)
SpiderNet_summary_LR_all = pd.concat(SpiderNet_summary_LR_list, axis=0)
linearmodel_summary_LR_all = pd.concat(linearmodel_summary_LR_list, axis=0)

SpiderNet_summary = SpiderNet_summary_all.copy()
linearmodel_summary = linearmodel_summary_all.copy()

spidernet_summary_path = base_run_dir / "insilico_spidernet_summary_leaveoneout.csv"
linearmodel_summary_path = base_run_dir / "insilico_linearmodel_summary_leaveoneout.csv"
spidernet_lr_summary_path = base_run_dir / "insilico_spidernet_lr_summary_leaveoneout.csv"
linearmodel_lr_summary_path = base_run_dir / "insilico_linearmodel_lr_summary_leaveoneout.csv"
insilico_seed_record_path = base_run_dir / "insilico_replacement_seed_record.csv"

SpiderNet_summary_all.to_csv(spidernet_summary_path)
linearmodel_summary_all.to_csv(linearmodel_summary_path)
SpiderNet_summary_LR_all.to_csv(spidernet_lr_summary_path)
linearmodel_summary_LR_all.to_csv(linearmodel_lr_summary_path)

if "insilico_seed_records" in globals() and len(insilico_seed_records) > 0:
    insilico_seed_record_df = pd.DataFrame(insilico_seed_records)
    insilico_seed_record_df.to_csv(insilico_seed_record_path, index=False)

# ------------------------------------------------------------
# Save observed and predicted LFC tables from the first run.
# These are reused by downstream scatter/barplot cells.
# ------------------------------------------------------------
lfc_cache_dir = base_run_dir / f"Cached_Tcell_LFC_for_scatter_seed{INSILICO_RANDOM_SEED}"
lfc_cache_dir.mkdir(parents=True, exist_ok=True)

observed_lfc_path = lfc_cache_dir / "Observed_Tcell_LFC_golden.csv"
spidernet_lfc_path = lfc_cache_dir / "Predicted_Tcell_LFC_SpiderNet.csv"
linearmodel_lfc_path = lfc_cache_dir / "Predicted_Tcell_LFC_LinearModel.csv"

observed_lr_lfc_path = lfc_cache_dir / "Observed_LRagg_LFC_golden.csv"
spidernet_lr_lfc_path = lfc_cache_dir / "Predicted_LRagg_LFC_SpiderNet.csv"
linearmodel_lr_lfc_path = lfc_cache_dir / "Predicted_LRagg_LFC_LinearModel.csv"

LFC_Tcell_golden_for_plot = LFC_Tcell_golden.copy()
LFC_LRagg_golden_for_plot = LFC_LRagg_golden.copy()

LFC_predicted_gene_spidernet_all = pd.concat(
    LFC_predicted_gene_spidernet_list,
    axis=1,
)
LFC_predicted_gene_linearmodel_all = pd.concat(
    LFC_predicted_gene_linearmodel_list,
    axis=1,
)

LFC_predicted_lr_spidernet_all = pd.concat(
    LFC_predicted_lr_spidernet_list,
    axis=1,
)
LFC_predicted_lr_linearmodel_all = pd.concat(
    LFC_predicted_lr_linearmodel_list,
    axis=1,
)

LFC_Tcell_golden_for_plot.to_csv(observed_lfc_path)
LFC_predicted_gene_spidernet_all.to_csv(spidernet_lfc_path)
LFC_predicted_gene_linearmodel_all.to_csv(linearmodel_lfc_path)

LFC_LRagg_golden_for_plot.to_csv(observed_lr_lfc_path)
LFC_predicted_lr_spidernet_all.to_csv(spidernet_lr_lfc_path)
LFC_predicted_lr_linearmodel_all.to_csv(linearmodel_lr_lfc_path)

print(f"Saved SpiderNet summary to: {spidernet_summary_path}")
print(f"Saved linear-model summary to: {linearmodel_summary_path}")
print(f"Saved SpiderNet LR summary to: {spidernet_lr_summary_path}")
print(f"Saved linear-model LR summary to: {linearmodel_lr_summary_path}")
if "insilico_seed_record_df" in globals():
    print(f"Saved in-silico replacement seed record to: {insilico_seed_record_path}")

print(f"Saved observed T-cell LFC to: {observed_lfc_path}")
print(f"Saved SpiderNet predicted T-cell LFC to: {spidernet_lfc_path}")
print(f"Saved LinearModel predicted T-cell LFC to: {linearmodel_lfc_path}")
print(f"Saved observed LR-aggregate LFC to: {observed_lr_lfc_path}")
print(f"Saved SpiderNet predicted LR-aggregate LFC to: {spidernet_lr_lfc_path}")
print(f"Saved LinearModel predicted LR-aggregate LFC to: {linearmodel_lr_lfc_path}")

SpiderNet_summary_all


## 11. Compare methods in a compact summary plot

In [ ]:
from matplotlib import rcParams

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
})

sns.set_style("white")

method_frames = []

SpiderNet_summary_plot = SpiderNet_summary.copy()
SpiderNet_summary_plot["Method"] = "SpiderNet"
SpiderNet_summary_plot["Gene"] = SpiderNet_summary_plot.index
method_frames.append(SpiderNet_summary_plot[["Gene", "Correlation", "Method"]])

linearmodel_summary_plot = linearmodel_summary.copy()
linearmodel_summary_plot["Method"] = "LinearModel"
linearmodel_summary_plot["Gene"] = linearmodel_summary_plot.index
method_frames.append(linearmodel_summary_plot[["Gene", "Correlation", "Method"]])

from run_benchmarks import saved_result
celcomen_summary_path = saved_result("celcomen_summary.csv")
if celcomen_summary_path.exists():
    celcomen_summary = pd.read_csv(celcomen_summary_path, index_col=0)
    celcomen_summary.columns = ["Correlation"]
    celcomen_summary["Method"] = "Celcomen"
    celcomen_summary["Gene"] = celcomen_summary.index
    method_frames.append(celcomen_summary[["Gene", "Correlation", "Method"]])

df_long = pd.concat(method_frames, ignore_index=True)
df_long.to_csv(base_run_dir / "Correlation_Summary_AllMethods.csv", index=False)

eps = 1e-8
df_wide = (
    df_long[df_long["Method"].isin(["SpiderNet", "LinearModel"])]
    .pivot_table(index="Gene", columns="Method", values="Correlation", aggfunc="mean")
)
df_wide["Ratio_SpiderNet_over_LinearModel_abs"] = (
    df_wide["SpiderNet"] / (np.abs(df_wide["LinearModel"]) + eps)
)
df_wide = df_wide.sort_values("Ratio_SpiderNet_over_LinearModel_abs", ascending=False)
df_wide.to_csv(base_run_dir / "Correlation_Ratio_SpiderNet_over_LinearModel.csv")

gene_order = df_wide.index.tolist()
df_long["Gene"] = pd.Categorical(df_long["Gene"], categories=gene_order, ordered=True)

palette = {
    "SpiderNet": "#cb1c68",
    "LinearModel": "#83499c",
}
if "Celcomen" in df_long["Method"].unique():
    palette["Celcomen"] = "#2a7bbf"

fig, ax = plt.subplots(figsize=(3.8, 2.6))

sns.barplot(
    data=df_long,
    x="Gene",
    y="Correlation",
    hue="Method",
    order=gene_order,
    palette=palette,
    errorbar=None,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T cell perturbation effect correlation\n(Observed vs Predicted)")
ax.set_title("")
ax.tick_params(axis="x", labelrotation=45)
plt.setp(ax.get_xticklabels(), ha="center")
ax.tick_params(axis="both", which="both", direction="out")
sns.despine(ax=ax, top=True, right=True)

ax.legend(
    title="",
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.18),
    ncol=min(3, df_long["Method"].nunique()),
    handlelength=1.2,
    columnspacing=1.2,
)

fig.tight_layout()
fig.savefig(base_run_dir / "Barplot_Correlation_by_Method_sorted_ratio_SpiderNet_over_LinearModel.png", dpi=600, bbox_inches="tight")
fig.savefig(base_run_dir / "Barplot_Correlation_by_Method_sorted_ratio_SpiderNet_over_LinearModel.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Load reusable LFC tables for downstream scatter plots
# ------------------------------------------------------------
# These files are generated in Section 10 when the first
# in-silico perturbation analysis is saved.
#
# Important:
# This cell should NOT reload models or recompute in-silico
# perturbations. It only loads the LFC tables computed earlier.
# ============================================================

from scipy.stats import spearmanr
import re

scatter_dir = base_run_dir / "Scatter_Observed_vs_Predicted_Tcell_effect_by_gene"
scatter_dir.mkdir(parents=True, exist_ok=True)

lfc_cache_dir = base_run_dir / f"Cached_Tcell_LFC_for_scatter_seed{INSILICO_RANDOM_SEED}"

observed_lfc_path = lfc_cache_dir / "Observed_Tcell_LFC_golden.csv"
spidernet_lfc_path = lfc_cache_dir / "Predicted_Tcell_LFC_SpiderNet.csv"
linearmodel_lfc_path = lfc_cache_dir / "Predicted_Tcell_LFC_LinearModel.csv"

observed_lr_lfc_path = lfc_cache_dir / "Observed_LRagg_LFC_golden.csv"
spidernet_lr_lfc_path = lfc_cache_dir / "Predicted_LRagg_LFC_SpiderNet.csv"
linearmodel_lr_lfc_path = lfc_cache_dir / "Predicted_LRagg_LFC_LinearModel.csv"


def _safe_filename(x):
    x = str(x)
    x = re.sub(r"[^\w\-.]+", "_", x)
    return x.strip("_")


def _load_lfc_table_if_needed(var_name, file_path):
    if var_name in globals():
        return globals()[var_name]

    if not file_path.exists():
        raise FileNotFoundError(
            f"Required cached LFC table does not exist: {file_path}\n"
            "Run Sections 9-10 first to compute and save the LFC tables. "
            "This plotting cell intentionally does not recompute in-silico perturbations."
        )

    table = pd.read_csv(file_path, index_col=0)
    globals()[var_name] = table
    return table


# Load only from memory or saved CSV files; no model prediction / no LFC recomputation.
LFC_Tcell_golden_for_plot = _load_lfc_table_if_needed(
    "LFC_Tcell_golden_for_plot",
    observed_lfc_path,
)
LFC_predicted_gene_spidernet_all = _load_lfc_table_if_needed(
    "LFC_predicted_gene_spidernet_all",
    spidernet_lfc_path,
)
LFC_predicted_gene_linearmodel_all = _load_lfc_table_if_needed(
    "LFC_predicted_gene_linearmodel_all",
    linearmodel_lfc_path,
)

# Optional LR-level LFC tables for future downstream plots.
if observed_lr_lfc_path.exists() or "LFC_LRagg_golden_for_plot" in globals():
    LFC_LRagg_golden_for_plot = _load_lfc_table_if_needed(
        "LFC_LRagg_golden_for_plot",
        observed_lr_lfc_path,
    )

if spidernet_lr_lfc_path.exists() or "LFC_predicted_lr_spidernet_all" in globals():
    LFC_predicted_lr_spidernet_all = _load_lfc_table_if_needed(
        "LFC_predicted_lr_spidernet_all",
        spidernet_lr_lfc_path,
    )

if linearmodel_lr_lfc_path.exists() or "LFC_predicted_lr_linearmodel_all" in globals():
    LFC_predicted_lr_linearmodel_all = _load_lfc_table_if_needed(
        "LFC_predicted_lr_linearmodel_all",
        linearmodel_lr_lfc_path,
    )

print("Loaded cached LFC tables from:", lfc_cache_dir)
print("Observed T-cell LFC shape:", LFC_Tcell_golden_for_plot.shape)
print("SpiderNet predicted T-cell LFC shape:", LFC_predicted_gene_spidernet_all.shape)
print("LinearModel predicted T-cell LFC shape:", LFC_predicted_gene_linearmodel_all.shape)


In [ ]:
# ============================================================
# Step 2. Plot one figure per perturbation gene.
# By default, this plots SpiderNet predictions.
# ============================================================

def _get_axis_limits(values, pad_frac=0.06):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    vmin = np.nanmin(values)
    vmax = np.nanmax(values)

    if np.isclose(vmin, vmax):
        pad = max(abs(vmin) * pad_frac, 1e-3)
    else:
        pad = pad_frac * (vmax - vmin)

    return vmin - pad, vmax + pad


def _plot_observed_vs_predicted_scatter(
    observed,
    predicted,
    perturb_gene,
    method_name,
    color,
    out_prefix,
    s=9,
    alpha=0.75,
):
    common_idx = observed.index.intersection(predicted.index)
    plot_df = pd.DataFrame({
        "Observed": observed.loc[common_idx].astype(float),
        "Predicted": predicted.loc[common_idx].astype(float),
    }).replace([np.inf, -np.inf], np.nan).dropna()

    if plot_df.shape[0] < 3:
        print(f"Skip {perturb_gene} | {method_name}: fewer than 3 valid points.")
        return np.nan, np.nan

    corr, pval = spearmanr(plot_df["Observed"], plot_df["Predicted"])

    fig, ax = plt.subplots(figsize=(2.6, 2.6))

    ax.scatter(
        plot_df["Observed"],
        plot_df["Predicted"],
        s=s,
        alpha=alpha,
        color=color,
        edgecolor="none",
    )

    # Regression line
    sns.regplot(
        data=plot_df,
        x="Observed",
        y="Predicted",
        scatter=False,
        color=color,
        line_kws={"linewidth": 1.0},
        ax=ax,
    )

    # Zero reference lines
    ax.axhline(0, color="0.75", linestyle="--", linewidth=0.7, zorder=0)
    ax.axvline(0, color="0.75", linestyle="--", linewidth=0.7, zorder=0)

    # Use independent x/y ranges from their own values
    x_min, x_max = _get_axis_limits(plot_df["Observed"].values)
    y_min, y_max = _get_axis_limits(plot_df["Predicted"].values)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.set_xlabel("Observed T-cell perturbation effect")
    ax.set_ylabel("Predicted T-cell perturbation effect")
    ax.set_title(f"{perturb_gene} | {method_name}", pad=5)

    # Corr annotation in the upper-right corner
    ax.text(
        0.97,
        0.97,
        f"ρ = {corr:.2f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=8,
    )

    ax.tick_params(axis="both", which="both", direction="out")
    sns.despine(ax=ax, top=True, right=True)

    fig.tight_layout()

    fig.savefig(f"{out_prefix}.png", dpi=600, bbox_inches="tight")
    fig.savefig(f"{out_prefix}.pdf", bbox_inches="tight")

    plt.show()
    plt.close(fig)

    return corr, pval


scatter_summary_rows = []

plot_gene_order = gene_order if "gene_order" in globals() else perturb_gene_OI

for perturb_gene_cur in plot_gene_order:
    if perturb_gene_cur not in LFC_Tcell_golden_for_plot.columns:
        print(f"Skip {perturb_gene_cur}: not found in observed LFC table.")
        continue

    if perturb_gene_cur not in LFC_predicted_gene_spidernet_all.columns:
        print(f"Skip {perturb_gene_cur}: not found in SpiderNet predicted LFC table.")
        continue

    out_prefix = scatter_dir / (
        f"Scatter_Observed_vs_Predicted_Tcell_effect_"
        f"SpiderNet_{_safe_filename(perturb_gene_cur)}"
    )

    corr, pval = _plot_observed_vs_predicted_scatter(
        observed=LFC_Tcell_golden_for_plot[perturb_gene_cur],
        predicted=LFC_predicted_gene_spidernet_all[perturb_gene_cur],
        perturb_gene=perturb_gene_cur,
        method_name="SpiderNet",
        color=palette.get("SpiderNet", "#cb1c68"),
        out_prefix=out_prefix,
    )

    scatter_summary_rows.append({
        "Gene": perturb_gene_cur,
        "Method": "SpiderNet",
        "SpearmanR": corr,
        "Pvalue": pval,
        "FigurePrefix": str(out_prefix),
    })

scatter_summary_df = pd.DataFrame(scatter_summary_rows)
scatter_summary_df.to_csv(
    scatter_dir / "Scatter_Observed_vs_Predicted_Tcell_effect_summary.csv",
    index=False,
)

display(scatter_summary_df)